# Seawater intrusion: a sharp interface (SWI) and variable density (BUY)

MODFLOW 6 offers two ways to simulate seawater in a coastal aquifer.

- The **Seawater Intrusion (SWI) Package** treats freshwater and seawater as separate fluids that meet at a sharp **interface**. One model layer is enough for an aquifer, because the interface can lie anywhere within a cell.
- The **Buoyancy (BUY) Package** couples a groundwater flow (GWF) model to a groundwater transport (GWT) model of the salt. The density of the water follows its salt concentration, so freshwater and seawater mix across a transition zone. The model needs many thin layers to resolve that zone, and it has to run through time until the salt stops moving.

This notebook solves one problem both ways: a confined coastal aquifer in cross section, with SWI in a single layer (a one-dimensional model) and BUY in 40 layers (a two-dimensional model). An analytical solution checks the SWI result, and a published correction for mixing explains the difference between the two.

> **Preview package.** SWI is not yet part of a MODFLOW 6 release. The class MODFLOW 6 includes it; see the [README](../../README.md).

## What this notebook covers

By the end of this notebook you will be able to:

- build a one-layer SWI model of a coastal aquifer and check it against the Dupuit–Ghyben–Herzberg solution,
- build a variable-density cross section with GWF, GWT, and BUY, and start it from the SWI interface,
- compare the SWI interface with the transition zone of the variable-density model,
- explain why mixing moves the saltwater toward the sea, and correct the sharp-interface answer for it.

## The problem

A confined aquifer 40 m thick runs 500 m from its inland edge to the sea. Its top is at sea level, so the sea fills the whole of the coastal face. Freshwater enters at the inland edge at 0.8 m³/d for each meter of coastline and leaves through the coastal face, flowing out over the seawater that lies beneath it.

| Property | Value |
|---|---|
| Length, inland edge to coast | 500 m |
| Thickness | 40 m, from 0 to -40 m |
| Hydraulic conductivity, $K$ | 10 m/d |
| Porosity | 0.2 |
| Freshwater inflow, $q$ | 0.8 m²/d (m³/d per meter of coastline) |
| Relative density difference, $\delta$ | 0.025 (seawater 1,025 kg/m³) |
| Salt concentration of seawater | 35 kg/m³ |

### The Dupuit–Ghyben–Herzberg solution

Where the flow is horizontal (the Dupuit assumption) and the saltwater stands still, the interface lies $h_f/\delta$ below sea level, 40 m for every meter of freshwater head (the Ghyben–Herzberg relation). The freshwater flows only above the interface, through a thickness $b = h_f/\delta$, so Darcy's law gives

$$q = K\,b\,\frac{dh_f}{dx'} = \frac{K}{\delta}\,h_f\,\frac{dh_f}{dx'},$$

where $x'$ is the distance inland from the coast. With $h_f = 0$ at the coast, the interface lies at a depth

$$b(x') = \sqrt{\frac{2\,q\,x'}{K\,\delta}},$$

and reaches the bottom of the aquifer, at the **toe** of the saltwater wedge, a distance

$$x'_\mathrm{toe} = \frac{K\,\delta\,H^2}{2\,q}$$

from the coast, where $H$ is the aquifer thickness. Here that is $10 \times 0.025 \times 40^2 / (2 \times 0.8) = 250$ m.

## Imports

In [ ]:
%matplotlib inline
import io
import pathlib as pl
import time

import flopy
import ipywidgets as widgets
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

width = 17.15 / 2.54  # figure width, inches

## Settings

All lengths are in meters and times in days. The grid has 100 columns 5 m wide. The SWI model has one layer; the variable-density model has 40 layers 1 m thick. `x_coast` is the distance of each cell center from the coast.

In [ ]:
length, thick = 500.0, 40.0  # m
k = 10.0  # m/d
porosity = 0.2
q = 0.8  # m2/d, freshwater inflow per meter of coastline
delta = 0.025  # relative density difference of seawater
c_sea = 35.0  # kg/m3, salt concentration of seawater
drhodc = 1000.0 * delta / c_sea  # density rise per kg/m3 of salt

delr, delz = 5.0, 1.0  # m
ncol, nlay = int(length / delr), int(thick / delz)
x = (np.arange(ncol) + 0.5) * delr  # cell centers, from the inland edge
x_coast = length - x

toe_dupuit = k * delta * thick**2 / (2 * q)
dupuit = -np.sqrt(np.minimum(2 * q * x_coast / (k * delta), thick**2))
print(f"Dupuit-Ghyben-Herzberg toe {toe_dupuit:.0f} m from the coast")

## The SWI model

One layer, one row, and 100 columns. A Well (**WEL**) in the first column brings in the freshwater. A General-Head Boundary (**GHB**) in the last column holds the freshwater head at 0 m, sea level, on the coastal face: its conductance is that of the half cell between the cell center and the face. With the head at sea level on the face, the interface reaches the top of the aquifer at the coast, as in the analytical solution.

SWI runs in single-fluid mode: the saltwater stands still with its head at sea level. The model is steady state. SWI needs the Newton formulation and a specific yield in the Storage (**STO**) Package, even for a steady-state run.

`swi_model()` builds and runs the model for a given relative density difference, so the same function can run it again later with a corrected value, and returns the interface elevation of each cell. Its optional arguments `k`, `q`, and `thick` change the hydraulic conductivity, the inflow, and the thickness of the aquifer, and `well` adds a pumping well, given as its column and pumping rate; the sliders at the end of the notebook use them.

In [ ]:
def swi_model(ws, delta_swi, k=k, q=q, thick=thick, well=None):
    sim = flopy.mf6.MFSimulation(sim_ws=ws, exe_name="mf6")
    flopy.mf6.ModflowTdis(sim)
    flopy.mf6.ModflowIms(
        sim,
        outer_dvclose=1e-8,
        outer_maximum=1000,
        inner_dvclose=1e-10,
        linear_acceleration="bicgstab",
    )
    gwf = flopy.mf6.ModflowGwf(sim, modelname="swi", newtonoptions="newton")
    flopy.mf6.ModflowGwfdis(
        gwf, nlay=1, nrow=1, ncol=ncol, delr=delr, delc=1.0, top=0.0, botm=-thick
    )
    flopy.mf6.ModflowGwfic(gwf, strt=0.5)
    flopy.mf6.ModflowGwfnpf(gwf, icelltype=0, k=k)
    flopy.mf6.ModflowGwfsto(
        gwf, iconvert=0, sy=porosity, ss=1e-5, steady_state={0: True}
    )
    wel_spd = [((0, 0, 0), q)]
    if well is not None:
        column, rate = well
        wel_spd.append(((0, 0, column), -rate))
    flopy.mf6.ModflowGwfwel(gwf, stress_period_data=wel_spd)
    cond = k * thick / (0.5 * delr)
    flopy.mf6.ModflowGwfghb(gwf, stress_period_data=[((0, 0, ncol - 1), 0.0, cond)])
    flopy.mf6.ModflowGwfswi(
        gwf, relative_density_difference=delta_swi, zeta_filerecord="swi.zta"
    )
    flopy.mf6.ModflowGwfoc(gwf, head_filerecord="swi.hds", saverecord=[("HEAD", "ALL")])
    sim.write_simulation(silent=True)
    success, buff = sim.run_simulation(silent=True)
    assert success, "MODFLOW 6 did not terminate normally"
    zeta = flopy.utils.HeadFile(pl.Path(ws) / "swi.zta", text="zeta").get_data()
    return zeta[0, 0]


def swi_toe(zeta, thick=thick):
    # distance from the coast of the farthest inland cell holding saltwater
    salt = zeta > -thick + 1e-6
    return x_coast[salt].max() if salt.any() else 0.0


start = time.perf_counter()
zeta_swi = swi_model("models/mf6-swi-buy-a/swi-1fluid", delta)
print(f"SWI run time {time.perf_counter() - start:.2f} s")
print(
    f"SWI toe {swi_toe(zeta_swi):.0f} m from the coast; "
    f"largest difference from the analytical interface "
    f"{np.abs(zeta_swi - dupuit).max():.2f} m"
)

Plot the SWI interface on the analytical solution. Freshwater is blue, saltwater orange.

In [ ]:
fresh_color, salt_color = "#9ecae1", "#fdae6b"

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.4 * width), layout="constrained")
    ax.fill_between(x_coast, -thick, 0.0, color=fresh_color, lw=0)
    ax.fill_between(x_coast, -thick, zeta_swi, color=salt_color, lw=0, step="mid")
    ax.step(x_coast, zeta_swi, where="mid", color="black", lw=1.0, label="SWI")
    ax.plot(x_coast, dupuit, color="red", ls="--", label="Dupuit-Ghyben-Herzberg")
    ax.set_xlim(length, 0.0)
    ax.set_ylim(-thick, 0.0)
    ax.set_xlabel("Distance from the coast (m)")
    ax.set_ylabel("Elevation (m)")
    ax.text(0.98 * length, -10, "freshwater", va="center")
    ax.text(40, -33, "saltwater", ha="right", va="center")
    flopy.plot.styles.graph_legend(ax=ax, loc="lower left", bbox_to_anchor=(0.02, 0.15))
    flopy.plot.styles.heading(ax=ax, heading="SWI and the analytical interface")
    plt.show()

**What to look for.** The SWI interface follows the analytical interface to within about 1 m over most of the wedge, and its toe lies 258 m from the coast, 8 m (less than two cells) beyond the analytical 250 m. The largest difference, 3.8 m, is in the last cell at the coast, where the analytical interface rises steeply to sea level over the last few meters, closer than one 5 m cell can follow. One layer and 100 cells give the wedge in a fraction of a second.

## The variable-density model

The same aquifer in 40 layers. The flow model gets the Buoyancy (**BUY**) Package, and a transport model carries the salt. BUY computes the density of the water in each cell from its salt concentration,

$$\rho = 1000 + \frac{\partial \rho}{\partial C}\,C,$$

with $\partial\rho/\partial C$ set so that seawater, 35 kg/m³ of salt, has a density of 1,025 kg/m³, the same $\delta$ of 0.025 as in the SWI model.

The boundaries match those of the SWI model, layer by layer:

- the WEL in the first column of each layer brings in 1/40 of the freshwater, with a concentration of 0;
- the GHB in the last column of each layer holds the coastal face at sea level, with a concentration of 35 kg/m³. BUY uses the concentration, through the GHB `concentration` auxiliary variable, to give the boundary the density of seawater, so the pressure of the sea on the face grows with depth.

The Source and Sink Mixing (**SSM**) package passes the concentrations of the WEL and GHB to the transport model. The transport model has a longitudinal dispersivity $\alpha_L$ of 1 m and a transverse dispersivity $\alpha_T$ of 0.1 m, and solves advection with the total-variation-diminishing (TVD) scheme, which keeps the transition zone from being smeared by the grid.

**Start from the SWI interface.** The salt takes years to reach a steady state, so the model runs through time. It starts with seawater below the SWI interface and freshwater above it. That start is close to the answer, so the run is short, and it shows how mixing moves the saltwater away from the sharp interface. Starting from an aquifer full of freshwater also works, but it takes longer, and the TVD scheme has trouble converging in the first time step, when seawater meets freshwater across the whole coastal face.

The run covers 20 years in 20-day time steps.

In [ ]:
def buy_model(ws, alpha_l, alpha_t, zeta_start, years=20.0, dt=20.0):
    sim = flopy.mf6.MFSimulation(sim_ws=ws, exe_name="mf6")
    nstp = int(years * 365.25 / dt)
    flopy.mf6.ModflowTdis(sim, perioddata=[(years * 365.25, nstp, 1.0)])
    botm = -delz * (np.arange(nlay) + 1.0)
    zc = -delz * (np.arange(nlay) + 0.5)  # cell centers

    gwf = flopy.mf6.ModflowGwf(sim, modelname="flow")
    ims = flopy.mf6.ModflowIms(
        sim,
        outer_dvclose=1e-6,
        outer_maximum=50,
        inner_maximum=200,
        inner_dvclose=1e-8,
        linear_acceleration="bicgstab",
        filename="flow.ims",
    )
    sim.register_ims_package(ims, ["flow"])
    flopy.mf6.ModflowGwfdis(
        gwf, nlay=nlay, nrow=1, ncol=ncol, delr=delr, delc=1.0, top=0.0, botm=botm
    )
    flopy.mf6.ModflowGwfic(gwf, strt=0.0)
    flopy.mf6.ModflowGwfnpf(gwf, icelltype=0, k=k)
    flopy.mf6.ModflowGwfsto(gwf, iconvert=0, ss=1e-5, transient={0: True})
    flopy.mf6.ModflowGwfbuy(
        gwf, packagedata=[(0, drhodc, 0.0, "transport", "concentration")]
    )
    wel = [((kk, 0, 0), q / nlay, 0.0) for kk in range(nlay)]
    flopy.mf6.ModflowGwfwel(
        gwf, auxiliary="concentration", pname="wel", stress_period_data=wel
    )
    cond = k * delz / (0.5 * delr)
    ghb = [((kk, 0, ncol - 1), 0.0, cond, c_sea) for kk in range(nlay)]
    flopy.mf6.ModflowGwfghb(
        gwf, auxiliary="concentration", pname="ghb", stress_period_data=ghb
    )
    flopy.mf6.ModflowGwfoc(
        gwf, head_filerecord="flow.hds", saverecord=[("HEAD", "LAST")]
    )

    gwt = flopy.mf6.ModflowGwt(sim, modelname="transport")
    imt = flopy.mf6.ModflowIms(
        sim,
        outer_dvclose=1e-4,
        outer_maximum=100,
        inner_maximum=300,
        inner_dvclose=1e-6,
        linear_acceleration="bicgstab",
        filename="transport.ims",
    )
    sim.register_ims_package(imt, ["transport"])
    flopy.mf6.ModflowGwtdis(
        gwt, nlay=nlay, nrow=1, ncol=ncol, delr=delr, delc=1.0, top=0.0, botm=botm
    )
    # seawater below the starting interface, freshwater above it
    strt = np.where(zc[:, None] < zeta_start[None, :], c_sea, 0.0)
    flopy.mf6.ModflowGwtic(gwt, strt=strt.reshape(nlay, 1, ncol))
    flopy.mf6.ModflowGwtmst(gwt, porosity=porosity)
    flopy.mf6.ModflowGwtadv(gwt, scheme="tvd")
    flopy.mf6.ModflowGwtdsp(gwt, alh=alpha_l, ath1=alpha_t, diffc=8.6e-5)
    flopy.mf6.ModflowGwtssm(
        gwt, sources=[("ghb", "aux", "concentration"), ("wel", "aux", "concentration")]
    )
    flopy.mf6.ModflowGwtoc(
        gwt,
        concentration_filerecord="transport.ucn",
        # about every two years, and at the end
        saverecord=[("CONCENTRATION", "FREQUENCY", 37), ("CONCENTRATION", "LAST")],
    )
    flopy.mf6.ModflowGwfgwt(sim, exgmnamea="flow", exgmnameb="transport")
    sim.write_simulation(silent=True)
    success, buff = sim.run_simulation(silent=True)
    assert success, "MODFLOW 6 did not terminate normally"
    ucn = flopy.utils.HeadFile(pl.Path(ws) / "transport.ucn", text="concentration")
    # relative concentration: 0 is freshwater, 1 seawater
    return np.array(ucn.get_times()) / 365.25, ucn.get_alldata()[:, :, 0, :] / c_sea


def buy_toe(c, level=0.5):
    # distance from the coast of the farthest inland bottom cell at or above level
    salt = c[-1] >= level
    return x_coast[salt].max() if salt.any() else 0.0


alpha_l, alpha_t = 1.0, 0.1  # m
start = time.perf_counter()
years, conc = buy_model("models/mf6-swi-buy-a/buy", alpha_l, alpha_t, zeta_swi)
print(f"BUY run time {time.perf_counter() - start:.1f} s")
for t, c in zip(years, conc):
    print(
        f"{t:5.1f} years: toe of the 50 percent seawater line "
        f"{buy_toe(c):.0f} m from the coast"
    )

**What to look for.** The run starts with the toe where SWI puts it, 258 m from the coast. Within two years the 50 percent seawater line at the bottom of the aquifer has fallen back to 222 m, and it keeps moving seaward more and more slowly: 182 m after 8 years and 178 m from 14 years on, where it stays. The variable-density model takes about 13 seconds for 20 years, a few hundred times as long as SWI.

## Compare the two models

Draw the relative concentration of the variable-density model at the end of the run, 0 for freshwater and 1 for seawater, with contours at 10, 50, and 90 percent seawater, and the SWI interface on top.

In [ ]:
botm = -delz * (np.arange(nlay) + 1.0)
z_centers = -delz * (np.arange(nlay) + 0.5)
cmap = mpl.colors.LinearSegmentedColormap.from_list("fs", [fresh_color, salt_color])


def on_bottom(zeta):
    # leave out the interface where it lies on the bottom, inland of the toe
    return np.where(zeta > -thick + 1e-6, zeta, np.nan)


with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.45 * width), layout="constrained")
    pc = ax.pcolormesh(
        np.r_[length, length - (np.arange(ncol) + 1) * delr],
        np.r_[0.0, botm],
        conc[-1],
        cmap=cmap,
        vmin=0.0,
        vmax=1.0,
    )
    cs = ax.contour(
        x_coast,
        z_centers,
        conc[-1],
        levels=[0.1, 0.5, 0.9],
        colors="0.25",
        linewidths=0.75,
    )
    ax.clabel(cs, fmt=lambda v: f"{v:.0%}", fontsize=7)
    # the SWI interface where there is saltwater; NaN along the bottom inland
    ax.step(
        x_coast,
        on_bottom(zeta_swi),
        where="mid",
        color="black",
        lw=1.5,
        label="SWI interface",
    )
    ax.set_xlim(length, 0.0)
    ax.set_ylim(-thick, 0.0)
    ax.set_xlabel("Distance from the coast (m)")
    ax.set_ylabel("Elevation (m)")
    fig.colorbar(pc, ax=ax, label="Fraction of seawater", shrink=0.9)
    flopy.plot.styles.graph_legend(ax=ax, loc="lower left", bbox_to_anchor=(0.02, 0.1))
    flopy.plot.styles.heading(
        ax=ax, heading=f"Variable density after {years[-1]:.0f} years, and SWI"
    )
    plt.show()

**What to look for.** The variable-density model has no sharp interface: the water changes from fresh to seawater across a transition zone 9 to 10 m thick, between the 10 and 90 percent lines. The SWI interface lies near the top of that zone, close to the 10 percent line, and its toe reaches 258 m from the coast, where the 50 percent line reaches only 178 m and the 90 percent line 148 m. At the coast the 50 percent line meets the coastal face about 5 m below sea level: the brackish water of the transition zone flows out to the sea through the upper part of the face, above the seawater flowing in below it.

## Why mixing moves the saltwater toward the sea

In SWI the saltwater stands still. In the variable-density model, dispersion mixes salt into the freshwater flowing above the transition zone, and that brackish water flows out to the sea. Seawater flows in along the bottom of the aquifer to replace the salt that is carried away, then up into the transition zone and back out to the sea. This circulation, driven by mixing, pushes the transition zone toward the coast.

Pool and Carrera (2011) compared sharp-interface and variable-density results over a wide range of aquifers and found that the sharp-interface solution matches the 50 percent seawater line when the relative density difference is reduced by a factor that depends on the transverse dispersivity and the aquifer thickness:

$$\delta^* = \delta\left[1 - \left(\frac{\alpha_T}{H}\right)^{1/6}\right].$$

The correction is large even for small dispersivities, because of the one-sixth power: with $\alpha_T = 0.1$ m and $H = 40$ m, $\delta^*$ is 0.63 $\delta$. Since the toe distance is proportional to $\delta$, the corrected toe lies at 63 percent of the sharp-interface distance.

Run the SWI model again with $\delta^*$, and compare.

> Pool, M., and Carrera, J., 2011, A correction factor to account for mixing in Ghyben-Herzberg and critical pumping rate approximations of seawater intrusion in coastal aquifers: Water Resources Research, v. 47, W05506, https://doi.org/10.1029/2010WR010256.

In [ ]:
factor = 1.0 - (alpha_t / thick) ** (1.0 / 6.0)
zeta_corrected = swi_model("models/mf6-swi-buy-a/swi-1fluid-corrected", factor * delta)
print(f"correction factor {factor:.3f}, corrected delta {factor * delta:.4f}")
print(f"toe, SWI:                      {swi_toe(zeta_swi):5.0f} m from the coast")
print(f"toe, SWI with corrected delta: {swi_toe(zeta_corrected):5.0f} m")
print(f"toe, variable density, 50 percent seawater: {buy_toe(conc[-1]):5.0f} m")

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.45 * width), layout="constrained")
    ax.pcolormesh(
        np.r_[length, length - (np.arange(ncol) + 1) * delr],
        np.r_[0.0, botm],
        conc[-1],
        cmap=cmap,
        vmin=0.0,
        vmax=1.0,
    )
    cs = ax.contour(
        x_coast, z_centers, conc[-1], levels=[0.5], colors="0.25", linewidths=0.75
    )
    ax.step(
        x_coast, on_bottom(zeta_swi), where="mid", color="black", lw=1.5, label="SWI"
    )
    ax.step(
        x_coast,
        on_bottom(zeta_corrected),
        where="mid",
        color="black",
        lw=1.5,
        ls="--",
        label="SWI, corrected for mixing",
    )
    ax.plot([], [], color="0.25", lw=0.75, label="50 percent seawater")
    ax.set_xlim(length, 0.0)
    ax.set_ylim(-thick, 0.0)
    ax.set_xlabel("Distance from the coast (m)")
    ax.set_ylabel("Elevation (m)")
    flopy.plot.styles.graph_legend(ax=ax, loc="lower left", bbox_to_anchor=(0.02, 0.1))
    flopy.plot.styles.heading(ax=ax, heading="Correcting SWI for mixing")
    plt.show()

**What to look for.** With $\delta^*$ the SWI toe moves from 258 m to 162 m from the coast, and the variable-density 50 percent line reaches 178 m. The corrected interface follows the 50 percent line closely all the way to the coast, a little seaward of it near the toe. Pool and Carrera fit their factor to many simulations, so a match within about 10 percent of the toe distance is what to expect. A sharp-interface model with no correction overstates how far the saltwater reaches inland, here by 45 percent.

### Try it: a smaller dispersivity

Set `alpha_l, alpha_t = 0.1, 0.01` in the cell that runs the variable-density model, and run that cell and every cell below it again. Before you run it, use the correction factor to predict where the 50 percent seawater line will reach the bottom of the aquifer. Does the transition zone get narrower? Does the toe move toward the SWI answer, and by how much? (With these dispersivities the correction factor is 0.75, and the corrected SWI toe lies 192 m from the coast. The variable-density toe moves from 178 m to 188 m, and the transition zone between the 10 and 90 percent lines thins only a little, to about 8 m: with 1 m layers, the grid itself smears the zone over a few cells.)

## Recap

You solved one coastal cross section two ways. The SWI model used a single layer and ran in a fraction of a second, and its interface matched the Dupuit–Ghyben–Herzberg solution. The variable-density model, with GWF, GWT, and BUY, needed 40 layers and a 20-year transient run, started from the SWI interface. It showed a transition zone instead of a sharp interface, and its 50 percent seawater line reached less far inland than the SWI interface, because mixing drives a circulation of seawater that pushes the saltwater toward the sea. Reducing the relative density difference in SWI by the Pool and Carrera (2011) factor brought the SWI toe close to the variable-density toe.

SWI is the quick tool for where the saltwater lies and how it moves in a regional model, with one layer per aquifer. The variable-density model is the tool when the mixing itself matters, such as the salinity a well will pump from the transition zone, at a much larger cost in cells and run time.

## Explore: change the aquifer with sliders

Each slider below changes one property of the aquifer or of a pumping well. When you release a slider, the notebook runs the SWI model twice, once with the relative density difference $\delta$ and once with the Pool and Carrera corrected $\delta^*$, and redraws the cross section. Each run takes a few hundredths of a second, so the picture updates almost at once. The variable-density model is too slow for this; the corrected SWI interface stands in for its 50 percent seawater line.

The plot shows:

- the SWI interface, with freshwater in blue and saltwater in orange;
- the Dupuit–Ghyben–Herzberg interface (red dashed line);
- the SWI interface with the corrected $\delta^*$ (black dashed line), which depends on the transverse dispersivity and the thickness;
- the pumping well (gray vertical line).

The well is screened over the whole aquifer, and its rate is set as a percent of the inflow, up to 90 percent, so that some freshwater still reaches the sea. The cross section is one meter wide, so the well stands for a row of wells along the coast, a line of pumping. The well takes freshwater out of the aquifer, and only the rest flows on to the sea. Seaward of the well the analytical interface is that of the smaller flow, $q - Q_w$, so the interface rises and the toe moves inland. The model has one layer, so it does not show the saltwater rising in a cone beneath the well; it shows the shift of the whole wedge.

The grid stays at 100 columns 5 m wide, so the aquifer is always 500 m long. If the saltwater reaches the inland edge, the analytical toe lies beyond the model.

Questions to try:

- Double $K$, then go back and halve $q$ instead. The analytical toe, $K\delta H^2/(2q)$, doubles both times. Does the SWI toe?
- Double the thickness. How far does the toe move?
- Set $\delta$ to 0.005, a brackish sea. Then set it to 0.04.
- Make the toe short, less than 50 m from the coast. How well does the single-layer SWI model follow the analytical interface when the wedge spans only a few cells?
- Put the well 100 m from the coast and pump half the inflow. How far inland does the toe move? Move the well inland past the toe: does the pumping still move the toe?
- Make the transverse dispersivity smaller. How close does the corrected interface come to the uncorrected one?


In [ ]:
explore_ws = "models/mf6-swi-buy-a/swi-1fluid-explore"
explore_bottom = -80.0  # lowest elevation shown, the bottom of the thickest aquifer
# the plot is drawn into an image that is replaced on each slider change,
# so the notebook never shows more than one copy of it
report = widgets.HTML()
picture = widgets.Image(format="png")


def show_wedge(k_ex, q_ex, delta_ex, thick_ex, alpha_t_ex, x_well, pumping):
    start = time.perf_counter()
    lines = []
    q_well = pumping / 100.0 * q_ex  # m2/d
    column = int(np.argmin(np.abs(x_coast - x_well)))
    model_args = {"k": k_ex, "q": q_ex, "thick": thick_ex, "well": (column, q_well)}
    factor_ex = 1.0 - (alpha_t_ex / thick_ex) ** (1.0 / 6.0)
    try:
        zeta_ex = swi_model(explore_ws, delta_ex, **model_args)
        zeta_cor = swi_model(explore_ws, factor_ex * delta_ex, **model_args)
    except AssertionError:
        report.value = (
            "MODFLOW 6 did not converge for these values. Move a slider and try again."
        )
        return
    run_time = time.perf_counter() - start

    # analytical interface: the flow to the sea is q - Q_w seaward of the well
    # and q inland of it
    x_w = x_coast[column]
    q_sea = q_ex - q_well
    flow_integral = q_sea * np.minimum(x_coast, x_w) + q_ex * np.maximum(
        x_coast - x_w, 0
    )
    dupuit_ex = -np.sqrt(np.minimum(2 * flow_integral / (k_ex * delta_ex), thick_ex**2))
    toe_integral = k_ex * delta_ex * thick_ex**2 / 2
    if q_sea * x_w >= toe_integral:
        toe_ex = toe_integral / q_sea
    else:
        toe_ex = x_w + (toe_integral - q_sea * x_w) / q_ex

    def toe_text(zeta):
        if zeta[0] > -thick_ex + 1e-6:
            return "reaches the inland edge"
        return f"{swi_toe(zeta, thick_ex):5.0f} m from the coast"

    lines.append(f"two SWI runs in {run_time:.2f} s")
    lines.append(f"well {x_w:.1f} m from the coast, pumping {q_well:.2f} m2/d")
    lines.append(f"toe, Dupuit-Ghyben-Herzberg: {toe_ex:5.0f} m from the coast")
    lines.append(f"toe, SWI:                    {toe_text(zeta_ex)}")
    lines.append(
        f"toe, SWI with corrected delta ({factor_ex:.2f} x delta): {toe_text(zeta_cor)}"
    )

    with flopy.plot.styles.USGSPlot():
        fig, ax = plt.subplots(figsize=(width, 0.5 * width), layout="constrained")
        ax.fill_between(x_coast, -thick_ex, 0.0, color=fresh_color, lw=0)
        ax.fill_between(x_coast, -thick_ex, zeta_ex, color=salt_color, lw=0, step="mid")
        ax.axhspan(explore_bottom, -thick_ex, color="0.85", lw=0)
        ax.step(x_coast, zeta_ex, where="mid", color="black", lw=1.0, label="SWI")
        ax.step(
            x_coast,
            np.where(zeta_cor > -thick_ex + 1e-6, zeta_cor, np.nan),
            where="mid",
            color="black",
            lw=1.5,
            ls="--",
            label="SWI, corrected for mixing",
        )
        ax.plot(
            x_coast, dupuit_ex, color="red", ls="--", label="Dupuit-Ghyben-Herzberg"
        )
        if q_well > 0:
            ax.plot(
                [x_w, x_w], [0.0, -thick_ex], color="0.3", lw=3, label="Pumping well"
            )
        ax.set_xlim(length, 0.0)
        ax.set_ylim(explore_bottom, 0.0)
        ax.set_xlabel("Distance from the coast (m)")
        ax.set_ylabel("Elevation (m)")
        fig.legend(loc="outside lower center", ncol=4, frameon=False)
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", dpi=110)
        plt.close(fig)
    picture.value = buffer.getvalue()
    report.value = "<pre>" + "\n".join(lines) + "</pre>"


style = {"description_width": "220px"}
layout = widgets.Layout(width="600px")
controls = {
    "k_ex": widgets.FloatLogSlider(
        value=k, min=0.0, max=2.0, step=0.05, description="hydraulic conductivity (m/d)"
    ),
    "q_ex": widgets.FloatLogSlider(
        value=q, min=-1.0, max=0.7, step=0.02, description="freshwater inflow (m2/d)"
    ),
    "delta_ex": widgets.FloatSlider(
        value=delta,
        min=0.005,
        max=0.04,
        step=0.001,
        readout_format=".3f",
        description="relative density difference",
    ),
    "thick_ex": widgets.FloatSlider(
        value=thick, min=10.0, max=80.0, step=5.0, description="aquifer thickness (m)"
    ),
    "alpha_t_ex": widgets.FloatLogSlider(
        value=0.1,
        min=-3.0,
        max=0.0,
        step=0.1,
        description="transverse dispersivity (m)",
    ),
    "x_well": widgets.FloatSlider(
        value=100.0,
        min=2.5,
        max=497.5,
        step=delr,
        readout_format=".1f",
        description="well distance from coast (m)",
    ),
    "pumping": widgets.FloatSlider(
        value=0.0, min=0.0, max=90.0, step=5.0, description="pumping (% of inflow)"
    ),
}


def update_wedge(change=None):
    show_wedge(**{name: control.value for name, control in controls.items()})


for control in controls.values():
    control.style = style
    control.layout = layout
    control.continuous_update = False  # run the model when the slider is released
    control.observe(update_wedge, names="value")
update_wedge()
display(widgets.VBox([*controls.values(), report, picture]))